In [ ]:
import datetime
from loguru import logger
from typing import Callable, Dict
from functools import wraps
from logger_wrapper import logger_wrapper
from read_excel_file import read_excel_file
from write_excel_file import write_excel_file
from helpers.create_predata import create_predata


def create_data_wrapper(postfix: bool = True, pre_data_info: Dict = None) -> Callable:
    '''
    Wrapper function for creating data.
    :param postfix: bool, whether to add postfix to the data.
    :param pre_data_info: Dict, information for creating predata. Format:
        { "sheet_name": {
            "file_path": str,
            "sheet_name": str,
            "create_data_func": Callable
            }
        }
    :return: Callable, wrapped function.
    '''
    def decorator(func: Callable) -> Callable:
        @logger_wrapper
        @wraps(func)
        def wrapper(
            file_path: str = "",
            sheet_name: str = "",
            is_index: bool = False
            ):
            logger.info(f"[{func.__name__} > {sheet_name}] Start creating data.")

            df = read_excel_file(file_path, sheet_name)
            if df is None:
                logger.error(f"[{func.__name__} > {sheet_name}] Can't read data.")
                return None

            if df.empty:
                logger.warning(f"[{func.__name__} > {sheet_name}] Data is empty.")

            if pre_data_info:
                logger.info(f"[{func.__name__} > {sheet_name}] Start creating predata.")
                predata_result = dict.fromkeys(pre_data_info.keys())
                for key, value in pre_data_info.items():
                    predata_result[key] = create_predata(**value)
                    if predata_result[key] is None or predata_result[key].empty:
                        logger.warning(f"[{func.__name__} > {sheet_name}] Create predata '{key}' failed.")
                        return None

            df = func(df, pre_data_info=predata_result)
            if df is None:
                logger.error(f"[{func.__name__} > {sheet_name}] Can't create data.")
                return None

            if df.empty:
                logger.warning(f"[{func.__name__} > {sheet_name}] Data is empty.")

            if postfix and "Created On" in df.columns:
                present_datetime: str = datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")
                author: str = "Rian mig test"
                df["Modified On"] = df["Created On"] = present_datetime
                df["Modified By"] = df["Created By"] = author

            write_excel_file(df, file_path, sheet_name, index=is_index)

            logger.info(f"[{func.__name__} > {sheet_name}] Finish creating data.")
            return df

        return wrapper

    return decorator


In [ ]:
# Usage
# file_path: str = r".\abc\file.xlsx"
# sheet_name: str = "SheetName"
# create_data_sample(file_path, sheet_name)

In [ ]:
# Example
import pandas as pd

AUTHOR_PREFIX = "Ri"

@create_data_wrapper(postfix=True)
def create_suspension_reason(df) -> pd.DataFrame | None:
    df = pd.DataFrame(columns=df.columns, index=[0])

    df.loc[0, "Status"] = ["Active", "Inactive"]

    df = df.explode("Status", ignore_index=True)

    df["Suspension reason name"] = AUTHOR_PREFIX + "SuspensionReasonName" + (df.index + 1).astype(str).str.zfill(2)

    df["Description"] = AUTHOR_PREFIX + "SuspensionReasonDescription" + (df.index + 1).astype(str).str.zfill(2)

    return df

In [ ]:
AUTHOR_PREFIX = "Ri"
START_DATE = "01/01/2025"
END_DATE = "31/12/2026"
DATE_FORMAT = "%d/%m/%Y"
DATETIME_FORMAT = "%d/%m/%Y %H:%M:%S"
AMOUNT = 6
MASTER_DATA_SETUP_INFO: Dict = {
    "file_path": r"C:\_My_job\_Code\_try python\gen_data\GeneratedDataV2\Data Template M02_Master Data Setup_C2R1.xlsx",
    "sheet_names": {
        "suspension_reason": "SuspensionReason",
        "internal_suspended_list": "InternalSuspendedList",
        "bank_information": "BankInformation",
        "industry_sector": "IndustrySector",
        "itm_industry": "ITMIndustry",
        "sector": "Sector",
        "service": "Service"
    }
}

In [ ]:
DEBTOR_TYPE = [
    "Individual",
    "Company"
]

PREDATA_INFO = {
    "suspension_reason": {
        "file_path": MASTER_DATA_SETUP_INFO.get("file_path"),
        "sheet_name": MASTER_DATA_SETUP_INFO.get("sheet_names").get("suspension_reason"),
        "create_data_func": create_suspension_reason
    }
}

@create_data_wrapper(
    postfix=True,
    pre_data_info=PREDATA_INFO
)
def create_internal_suspended_list(df: pd.DataFrame, pre_data_info: pd.DataFrame) -> pd.DataFrame | None:
    df_suspension_reason = pre_data_info.get("suspension_reason")

    df = pd.DataFrame(columns=df.columns, index=[0])

    df.loc[0, "Debtor type"] = DEBTOR_TYPE

    df.loc[0,"Suspended reason"] = df_suspension_reason["Suspension reason name"].tolist()

    df.loc[0, "Source"] = ["Manual", "Auto"]

    df.loc[0, "Status"] = ["Active", "Inactive"]

    df = df.explode("Debtor type", ignore_index=True).explode("Status", ignore_index=True).explode("Source", ignore_index=True).explode("Suspended reason", ignore_index=True)

    df["ID"] = df.index.map(lambda x: f"IS{str(x + 10_001)}")

    df["NRIC/FIN"] = df.apply(lambda x: generate_nric("S") if x["Debtor type"] == "Individual" else "", axis=1)

    df["UEN"] = df.apply(lambda x: generate_uen() if x["Debtor type"] == "Company" else "", axis=1)

    df["Effective from"] = START_DATE

    df["Effective to"] = df.index.map(lambda x: END_DATE if x % 2 == 0 else "")

    df["Bad debt write-off amount"] = np.random.uniform(0, 10000, df.shape[0]).round(2)

    df["Remarks"] = df.index.map(lambda x: f"{AUTHOR_PREFIX} Internal suspended list {str(x + 1).zfill(2)}")

    return df